# DCT Laboratory — Volume I, Chapter 9
## Enterprise State Architecture
**Seed `26109`** · Companion to Chapter 9 and **AXIOM Lab 1.9** (module **AXIOM-09**) · Mirrored in `DCT_V1_Ch09_Lab.xlsx`

This laboratory reproduces **Worked Example 9.1, *How Fast Can a Demand Shock Reach the Covenant?***, with the
book's declarations, and works the five steps of AXIOM Lab 1.9, *Wiring Meridian*: declare the architecture, find
the core, send a shock, test the summary map, and audit the constraints. Every answer is a computation on the
declared wiring: distances by breadth-first search, feedback cores by Tarjan's algorithm, routes by matrix powers,
cuts by augmenting paths, and feasibility by the cycle test. None of the book's numbers is random; the lab's seed
26109 is used only for fresh draws (sensitivities on the declared arrows, random constraint families), labelled as
such. Every Meridian number below is the book's or follows from its declarations (Worked Example 9.1, Table 9.7,
Figures 9.1–9.3, Section 9.8, Online Appendix 9.B.5–9.B.7); the final cell checks the printed ones. Money is in
\$ million.

## The declarations (Worked Example 9.1, Step 1; Tables 9.1–9.3)

**Seven components** (Table 9.1): the three divisions, Industrial Systems (IND), Digital (DIG) and Materials (MAT);
shared engineering (ENG); treasury (TRS); the market (MKT); and governance (GOV). The model runs in quarters.

**Fourteen typed arrows** (Figure 9.1): demand from the market to each division; cash from each division to
treasury; funding from treasury to Industrial Systems and Digital (Materials funds its own capital spending);
capability both ways between engineering and each of Industrial Systems and Digital; and policy from governance to
treasury and Digital. An arrow $u \to v$ asserts that $v$'s next state reads $u$'s state; a missing arrow asserts
that it does not.

**The constraints, with their scopes** (Table 9.3). Treasury keeps the group's books (cash, debt, and the EBITDA the
divisions report along the cash arrows), so the loan covenant, net leverage at most 4.5 times EBITDA (Table 5.4), and
the board's liquidity floor of \$350 million are local constraints of treasury; plant capacity is local to Industrial
Systems. No component collects strategic risk: the ceiling of 75 reads the three divisions' risk positions directly,
so it couples IND, DIG and MAT. The digital build's requirement that workforce capability stay at 55 or above
(Worked Example 3.1) couples engineering and Digital. The liquidity floor and the risk ceiling are board policies
the company can revise; the covenant only the lenders can change. Index bounds are local too.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mp
from matplotlib.lines import Line2D
from itertools import combinations
from scipy.optimize import minimize_scalar
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26109          # the lab's seed: fresh draws only, labelled as such

# Table 9.1: the seven components, listed by causal layer as in the book's engine
NODES = ["MKT", "GOV", "MAT", "IND", "DIG", "ENG", "TRS"]
NAME = {"MKT": "Market", "GOV": "Governance", "MAT": "Materials", "IND": "Industrial Systems",
        "DIG": "Digital", "ENG": "Shared engineering", "TRS": "Treasury"}
BOOK_ORDER = ["IND", "DIG", "MAT", "ENG", "TRS", "MKT", "GOV"]          # the order of Tables 9.1 and 9.7
TWO_BOX = {"MKT": "market", "IND": "divisions", "DIG": "divisions", "MAT": "divisions",
           "ENG": "center", "TRS": "center", "GOV": "center"}           # the two-box map (Figure 9.2)
IDX = {v: i for i, v in enumerate(NODES)}

# Figure 9.1 (Step 1): fourteen typed arrows (from, to, type); u -> v: v's next state reads u's state
ARROWS = [("MKT", "IND", "demand"), ("MKT", "DIG", "demand"), ("MKT", "MAT", "demand"),
          ("IND", "TRS", "cash"), ("DIG", "TRS", "cash"), ("MAT", "TRS", "cash"),
          ("TRS", "IND", "funding"), ("TRS", "DIG", "funding"),
          ("ENG", "IND", "capability"), ("IND", "ENG", "capability"),
          ("ENG", "DIG", "capability"), ("DIG", "ENG", "capability"),
          ("GOV", "TRS", "policy"), ("GOV", "DIG", "policy")]
CLASS = {"demand": "dynamic", "cash": "resource flow", "funding": "resource flow",
         "capability": "resource flow", "policy": "informational"}          # Table 9.2
DECLARED = {(u, v) for u, v, _ in ARROWS}

# Tables 9.1 and 9.3: the constraints, their scopes, and who can change them ($ million)
CONSTRAINTS = [("loan covenant: net leverage at most 4.5x EBITDA", ["TRS"], "only the lenders can change it"),
               ("liquidity floor: at least $350 million", ["TRS"], "board policy, revisable"),
               ("plant capacity", ["IND"], "a physical limit (Table 5.4)"),
               ("strategic-risk ceiling of 75", ["IND", "DIG", "MAT"], "board policy, revisable"),
               ("workforce capability at least 55", ["ENG", "DIG"], "the digital build's requirement")]
CEILING, RISK_ERROR = 75.0, 6.0         # the risk ceiling; the risk committee's measurement error (Table 5.2)

def scope_class(scope):
    return "local" if len(scope) == 1 else ("pairwise coupling" if len(scope) == 2 else "group coupling")

# ---- graph computations on any list of arrows (the book's figures call networkx; the same algorithms, written out)
def successors(arrows, nodes=NODES):
    out = {v: [] for v in nodes}
    for u, v, *_ in arrows:
        out[u].append(v)                       # in the order the arrows are declared
    return out

def adjacency(arrows, nodes=NODES):
    A = np.zeros((len(nodes), len(nodes)), dtype=int)
    for u, v, *_ in arrows:
        A[nodes.index(u), nodes.index(v)] = 1
    return A

def bfs(arrows, source, nodes=NODES):
    # breadth-first search: graph distances from the source and a shortest route to each component reached
    out, dist, parent, frontier = successors(arrows, nodes), {source: 0}, {source: None}, [source]
    while frontier:
        nxt = []
        for u in frontier:
            for v in out[u]:
                if v not in dist:
                    dist[v], parent[v] = dist[u] + 1, u
                    nxt.append(v)
        frontier = nxt
    return dist, parent

def route(parent, v):
    r = [v]
    while parent[r[-1]] is not None:
        r.append(parent[r[-1]])
    return r[::-1]

def tarjan(arrows, nodes=NODES):
    # Tarjan's algorithm (1972): the strongly connected components, emitted in reverse topological order
    out, index, low, on, stack, comps = successors(arrows, nodes), {}, {}, set(), [], []
    def visit(v):
        index[v] = low[v] = len(index); stack.append(v); on.add(v)
        for w in out[v]:
            if w not in index:
                visit(w); low[v] = min(low[v], low[w])
            elif w in on:
                low[v] = min(low[v], index[w])
        if low[v] == index[v]:
            comp = []
            while True:
                w = stack.pop(); on.discard(w); comp.append(w)
                if w == v:
                    break
            comps.append([x for x in nodes if x in comp])
    for v in nodes:
        if v not in index:
            visit(v)
    return comps

def condensation(arrows, nodes=NODES):
    # Proposition 9.3: one node per strongly connected component; level 1 if no arrow enters, else 1 + max
    comps = tarjan(arrows, nodes)
    of = {v: i for i, c in enumerate(comps) for v in c}
    links = sorted({(of[u], of[v]) for u, v, *_ in arrows if of[u] != of[v]})
    level = {}
    for i in reversed(range(len(comps))):          # reversed Tarjan order is a topological order
        level[i] = 1 + max((level[a] for a, b in links if b == i), default=0)
    return comps, of, links, level

def simple_cycles(arrows, nodes=NODES):
    # every directed loop once, started at its first member in the node order
    out, found = successors(arrows, nodes), []
    def extend(path):
        for w in out[path[-1]]:
            if w == path[0]:
                found.append(path[:])
            elif w not in path and nodes.index(w) > nodes.index(path[0]):
                extend(path + [w])
    for s in nodes:
        extend([s])
    return found

def simple_paths(arrows, s, t, nodes=NODES):
    out, found = successors(arrows, nodes), []
    def extend(path):
        for w in out[path[-1]]:
            if w == t:
                found.append(path + [w])
            elif w not in path:
                extend(path + [w])
    extend([s])
    return found

def pieces(links, nodes=NODES):
    # separate pieces of the graph when arrow directions are ignored
    nbr = {v: set() for v in nodes}
    for u, v, *_ in links:
        nbr[u].add(v); nbr[v].add(u)
    out, seen = [], set()
    for s in nodes:
        if s not in seen:
            piece, stack = {s}, [s]
            while stack:
                for w in nbr[stack.pop()] - piece:
                    piece.add(w); stack.append(w)
            seen |= piece; out.append([v for v in nodes if v in piece])
    return out

print(f"{len(NODES)} components, {len(ARROWS)} typed arrows: " + ", ".join(
    f"{sum(t == k for *_, t in ARROWS)} {k}" for k in ("demand", "cash", "funding", "capability", "policy")))
for name, scope, note in CONSTRAINTS:
    print(f"  {name:48s} scope {{{', '.join(scope)}}}: {scope_class(scope)}; {note}")

## Panel 1 — Declare the architecture (Worked Example 9.1, Step 1; Theorem 9.1; Tables 9.1–9.2: 7 components, 14 arrows)

AXIOM Lab 1.9, step 1: inspect the seven components, the fourteen typed arrows and the constraints with their
scopes, and for each arrow name the reading it carries and check that its source exposes that reading
(Definition 9.2). Table 9.1 names two interfaces, treasury's funding capacity and engineering's
workforce-capability index, and Step 1 a third: the divisions report cash and EBITDA along the cash arrows. Every
arrow is interface-backed, so the components assemble into one company model (Theorem 9.1(ii)); were treasury not to
expose funding capacity, two arrows would lose their reading and the updates of Industrial Systems and Digital would
be undefined (Proposition 9.5). Theorem 9.1(i) reads the wiring back from the dynamics: in a model assembled with
sensitivities drawn with the lab's seed, changing one component at a time and watching which next states move
recovers exactly the fourteen declared arrows. The 28 missing arrows are declarations too: nothing points into the
market or into governance, and no funding arrow reaches Materials.

In [ ]:
# interfaces: Table 9.1 names treasury's (funding capacity) and engineering's (the workforce-capability index);
# Step 1 names the divisions' cash arrows (cash and EBITDA); the other readings are named by their arrow's type
INTERFACE = {"MKT": {"demand"}, "GOV": {"policy"}, "MAT": {"cash and EBITDA"},
             "IND": {"cash and EBITDA", "capability"}, "DIG": {"cash and EBITDA", "capability"},
             "ENG": {"workforce-capability index"}, "TRS": {"funding capacity"}}

def carries(u, t):
    # the reading an arrow of type t leaving u carries
    if t == "capability":
        return "workforce-capability index" if u == "ENG" else "capability"
    return {"demand": "demand", "cash": "cash and EBITDA", "funding": "funding capacity", "policy": "policy"}[t]

def backed(arrows, interface):
    return [a for a in arrows if carries(a[0], a[2]) in interface[a[0]]]

print(f"{'arrow':12s} {'type':11s} {'class (Table 9.2)':18s} {'reading carried':28s} source exposes it")
for u, v, t in ARROWS:
    r = carries(u, t)
    print(f"{u} -> {v}   {t:11s} {CLASS[t]:18s} {r:28s} {'yes' if r in INTERFACE[u] else 'NO'}")
BACKED = len(backed(ARROWS, INTERFACE))
print(f"interface-backed: {BACKED} of {len(ARROWS)} arrows, so the components assemble into one model (Theorem 9.1(ii))")
lost = [a for a in ARROWS if a not in backed(ARROWS, dict(INTERFACE, TRS=set()))]
print(f"were treasury not to expose funding capacity: {len(lost)} arrows unbacked ({', '.join(f'{u}->{v}' for u, v, _ in lost)});"
      f" the next states of {' and '.join(v for _, v, _ in lost)} would be undefined (Proposition 9.5)")

# Theorem 9.1(i)-(iii) on an assembled model: each component's next state is its own carry-over plus a
# positive sensitivity to each in-neighbor (fresh draws, lab seed)
rng = np.random.default_rng(SEED)
SENS = {(u, v): float(s) for (u, v, _), s in zip(ARROWS, rng.uniform(0.2, 0.8, len(ARROWS)))}
KEEP = {v: float(c) for v, c in zip(NODES, rng.uniform(0.3, 0.9, len(NODES)))}

def assemble(sens=SENS, keep=KEEP):
    M = np.diag([keep[v] for v in NODES])
    for (u, v), s in sens.items():
        M[IDX[v], IDX[u]] = s                  # v's next state reads u's: M[v, u] > 0 exactly on an arrow u -> v
    return M

M = assemble()
x0 = rng.normal(size=len(NODES))
CANON = []
for u in NODES:                                # change u alone; an arrow u -> v wherever v's next state moves
    x1 = x0.copy(); x1[IDX[u]] += 1.0
    CANON += [(u, v) for v in NODES if v != u and abs((M @ x1 - M @ x0)[IDX[v]]) > 1e-12]
print(f"\ncanonical graph of the assembled model (sensitivities drawn with seed {SEED}): {len(CANON)} arrows, "
      f"{'equal to' if set(CANON) == DECLARED else 'NOT equal to'} the declared wiring (Theorem 9.1(iii))")

A = adjacency(ARROWS)
n = len(NODES)
MISSING = n * (n - 1) - int(A.sum())
print(f"\nmissing arrows: {MISSING} of the {n * (n - 1)} possible, each a claim of independence")
for v in BOOK_ORDER:
    ins = [u for u in BOOK_ORDER if A[IDX[u], IDX[v]]]
    print(f"  into {v}: {len(ins)} of {n - 1}" + (f" (from {', '.join(ins)})" if ins else ": nothing reaches it directly"))
FUND_MAT = sum(1 for u, v, t in ARROWS if v == "MAT" and t == "funding")
print(f"funding arrows into Materials: {FUND_MAT}: the architecture asserts that Materials funds itself")

## Panel 2 — Find the core (Worked Example 9.1, Step 2; Theorem 9.2(iii); Proposition 9.3; Figure 9.1: one core of four, six loops, three layers)

AXIOM Lab 1.9, step 2, the topology analyzer. Tarjan's algorithm finds the strongly connected components, and one has
more than one member, {IND, DIG, ENG, TRS}: Meridian's feedback core. It contains six distinct feedback loops: four
two-way relationships, cash and funding between treasury and each of Industrial Systems and Digital and capability
between engineering and each of them, and two loops that run through all four members. Collapsing each strongly
connected component to one node gives the condensation, which has no loops (Proposition 9.3). Its layers are the
drivers (the market and governance, which drive the core and are driven by nothing), a feeder (Materials, which
receives demand and sends cash but hears nothing back) and the core; the longest chain of one-way arrows has two
steps. Compare with the org chart: the core cuts across it, two divisions and two corporate functions, with the third
division outside. Inside a core, mutual influence is the rule but not a law: in Online Appendix 9.B.1 two offsetting
channels let a transfer-pricing rule leave consolidated profit untouched, although consolidated profit moves the rule.

In [ ]:
COMPS, OF, CLINKS, LEVEL = condensation(ARROWS)
CORES = [c for c in COMPS if len(c) > 1]
CORE = CORES[0]
LAYER_NAME = {1: "drivers", 2: "feeder", 3: "feedback core"}
print("strongly connected components (Tarjan):", "  ".join("{" + ", ".join(c) + "}" for c in COMPS))
print(f"feedback cores (more than one member): {len(CORES)}, {{{', '.join(CORE)}}} with {len(CORE)} members")
LOOPS = simple_cycles(ARROWS)
TWO_WAY = [c for c in LOOPS if len(c) == 2]
print(f"directed loops: {len(LOOPS)}, every one inside the core: {all(set(c) <= set(CORE) for c in LOOPS)}")
for c in sorted(LOOPS, key=len):
    print("   " + " -> ".join(c + [c[0]]))
print(f"  {len(TWO_WAY)} two-way relationships and {len(LOOPS) - len(TWO_WAY)} loops through all four members")
print("condensation:", ", ".join("{" + ", ".join(COMPS[a]) + "} -> {" + ", ".join(COMPS[b]) + "}" for a, b in CLINKS))
for lv in sorted(set(LEVEL.values())):
    print(f"  layer {lv}, {LAYER_NAME[lv]}: " + "  ".join("{" + ", ".join(COMPS[i]) + "}" for i in LEVEL if LEVEL[i] == lv))
LAYERS, DEPTH = max(LEVEL.values()), max(LEVEL.values()) - 1
print(f"{LAYERS} layers; depth of the condensation (its longest chain of one-way arrows): {DEPTH}")
by_box = {b: [v for v in BOOK_ORDER if v in CORE and TWO_BOX[v] == b] for b in ("divisions", "center")}
print(f"the core against the org chart: two divisions ({' and '.join(by_box['divisions'])}) and two corporate functions"
      f" ({' and '.join(by_box['center'])}); Materials, the third division, sits upstream of the core as its feeder")

# Online Appendix 9.B.1: a transfer-pricing rule u, two divisions' reported profits a and b that trade with
# each other, consolidated profit v: a' = u, b' = -u, v' = a + b, u' = u + c v
OFF = [("u", "a"), ("u", "b"), ("a", "v"), ("b", "v"), ("v", "u")]
def offsetting(a, b, v, u, c=0.5, quarters=12):
    path = []
    for _ in range(quarters + 1):
        path.append((a, b, v, u)); a, b, v, u = u, -u, a + b, u + c * v
    return np.array(path)
base = offsetting(1.0, 2.0, 3.0, 4.0)
rule_up = offsetting(1.0, 2.0, 3.0, 5.0)          # raise the rule by 1
profit_up = offsetting(1.0, 2.0, 4.0, 4.0)        # raise consolidated profit by 1
OFF_CORE = len(tarjan(OFF, ["u", "a", "b", "v"]))
OFF_MOVE = float(abs(rule_up[:, 2] - base[:, 2]).max())
OFF_BACK = float(profit_up[1, 3] - base[1, 3])
print(f"\nOnline Appendix 9.B.1: the four components form {OFF_CORE} strongly connected component; raising the rule by 1"
      f" moves consolidated profit by {OFF_MOVE:.0f} in twelve quarters, while raising consolidated profit by 1 raises"
      f" the next rule by c = {OFF_BACK:.1f}")

# the drawing: one column of bands per causal layer, as computed above; arrows styled by type (Figure 9.1)
POS = {"MKT": (0.95, 3.0), "GOV": (0.95, 0.55), "MAT": (4.05, 4.15), "IND": (6.95, 3.55),
       "ENG": (8.95, 2.05), "DIG": (6.95, 0.55), "TRS": (11.95, 2.05)}
LABEL = {"MKT": "Market\n(MKT)", "GOV": "Governance\n(GOV)", "MAT": "Materials\n(MAT)",
         "IND": "Industrial\nSystems (IND)", "DIG": "Digital\n(DIG)", "ENG": "Engineering\n(ENG)", "TRS": "Treasury\n(TRS)"}
STYLE = {"demand": ("#E65100", "-"), "cash": ("#2E7D32", "-"), "funding": ("#2E7D32", "--"),
         "capability": ("#5E35B1", "-"), "policy": ("#616161", ":")}
BEND = {("MAT", "TRS"): -0.3, ("GOV", "TRS"): 0.3, ("MKT", "MAT"): -0.1, ("MKT", "DIG"): 0.06,
        ("IND", "TRS"): -0.2, ("TRS", "IND"): 0.05, ("DIG", "TRS"): 0.2, ("TRS", "DIG"): -0.05}
BAND = {1: (-0.2, 2.1), 2: (2.85, 5.25), 3: (5.6, 13.35)}
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.set_xlim(-0.3, 13.45); ax.set_ylim(-0.75, 5.3); ax.axis("off")
for lv, (a0, a1) in BAND.items():
    ax.add_patch(mp.FancyBboxPatch((a0, -0.45), a1 - a0, 5.55, boxstyle="round,pad=0.02,rounding_size=0.15",
                                   fc="#EEF3F9" if lv == 3 else "#F4F4F4", ec="none", zorder=0))
    ax.text((a0 + a1) / 2, 4.85, f"layer {lv}: {LAYER_NAME[lv]}", ha="center", va="center", fontsize=8,
            fontweight="bold", color="#1F4E79" if lv == 3 else "#616161")
ELL = {}
for v, (x, y) in POS.items():
    ELL[v] = mp.Ellipse((x, y), 2.05, 0.86, fc="white", ec="#1F4E79" if v in CORE else "#1A1A1A", lw=1.1, zorder=3)
    ax.add_patch(ELL[v])
    ax.text(x, y, LABEL[v], ha="center", va="center", fontsize=7, zorder=4, linespacing=1.0)
for u, v, t in ARROWS:
    col, ls = STYLE[t]
    rad = BEND.get((u, v), -0.22 if (v, u) in DECLARED else 0.0)
    ax.add_patch(mp.FancyArrowPatch(POS[u], POS[v], patchA=ELL[u], patchB=ELL[v], connectionstyle=f"arc3,rad={rad}",
                                    arrowstyle="-|>", mutation_scale=8, lw=1.0, color=col, linestyle=ls,
                                    shrinkA=1, shrinkB=1.5, zorder=2))
ax.legend([Line2D([], [], color=c, ls=s, lw=1.2) for c, s in STYLE.values()], list(STYLE), loc="lower center",
          ncol=5, frameon=False, fontsize=7.5, bbox_to_anchor=(0.5, -0.08), handlelength=2.6)
ax.set_title("Meridian's wiring: drivers, a feeder and one feedback core (seed 26109)", fontsize=10)
plt.tight_layout(); plt.show()

## Panel 3 — Put floors under arrival times (Worked Example 9.1, Step 3; Theorem 9.2(i)–(ii); Table 9.7: the covenant no earlier than the second quarter-end, three cuts)

Worked Example 9.1, Step 3, and AXIOM Lab 1.9, step 3, first part. Breadth-first search from the market gives each
component's graph distance, the earliest quarter-end at which a demand shock can change it (Table 9.7): the three
divisions in the first quarter, engineering and treasury in the second, and governance never, because no path leads
there. The covenant is a reading of treasury, so by Theorem 9.2(ii) it cannot move before the second quarter-end
after the shock: a floor, not a forecast. Five distinct paths lead from the market to treasury, three of two steps,
one through each division, and two of four steps, through engineering. Three of them have no arrow in common, so at
least three arrows must be cut to separate treasury from the market (the fewest cuts equal the most paths that share
no arrow, Ford and Fulkerson): for instance the three cash arrows into treasury, or the three demand arrows out of
the market. No cut of one or two arrows does it.

In [ ]:
DIST, PARENT = bfs(ARROWS, "MKT")
ROW97 = {"IND": "Industrial Systems (IND)", "DIG": "Digital (DIG)", "MAT": "Materials (MAT)",
         "ENG": "Shared engineering (ENG)", "TRS": "Treasury (TRS), covenant", "GOV": "Governance (GOV)"}
TABLE97 = []
for v in ("IND", "DIG", "MAT", "ENG", "TRS", "GOV"):
    if v in DIST:
        via = [u for u in BOOK_ORDER if DIST.get(u) == DIST[v] - 1 and (u, v) in DECLARED and u != PARENT[v]]
        r = " → ".join(route(PARENT, v)) + (f" (also via {', '.join(via)})" if via else "")
        TABLE97.append((ROW97[v], str(DIST[v]), r))
    else:
        TABLE97.append((ROW97[v], "never", "no path"))
print("Table 9.7: the earliest quarter-end at which a demand shock can change each component")
print(f"  {'component':26s} {'earliest quarter':>17s}   a shortest route")
for row in TABLE97:
    print(f"  {row[0]:26s} {row[1]:>17s}   {row[2]}")
for k in (1, 2, 3):
    print(f"after {k} quarter{'s' if k > 1 else ''}: the shock can have reached "
          + ", ".join(v for v in BOOK_ORDER if v != "MKT" and DIST.get(v, 99) <= k))
print("never reached:", ", ".join(v for v in NODES if v not in DIST), "; the covenant cannot move before quarter", DIST["TRS"])

PATHS = simple_paths(ARROWS, "MKT", "TRS")
print(f"\n{len(PATHS)} distinct paths from the market to treasury:")
for p in sorted(PATHS, key=len):
    print(f"  {len(p) - 1} steps: {' -> '.join(p)}")
TWO_STEP = [p for p in PATHS if len(p) == 3]
used = [a for p in TWO_STEP for a in zip(p, p[1:])]
print(f"the {len(TWO_STEP)} two-step paths use {len(set(used))} different arrows: no arrow in common")

def max_flow(arrows, s, t):
    # augmenting paths with unit capacities: the most paths sharing no arrow, and a cut of the same size
    cap = {}
    for u, v, *_ in arrows:
        cap[(u, v)] = cap.get((u, v), 0) + 1
    flow = 0
    while True:
        parent, queue = {s: None}, [s]
        while queue and t not in parent:
            u = queue.pop(0)
            for (a, b), c in list(cap.items()):
                if a == u and c > 0 and b not in parent:
                    parent[b] = u; queue.append(b)
        if t not in parent:
            break
        v = t
        while parent[v] is not None:
            u = parent[v]; cap[(u, v)] -= 1; cap[(v, u)] = cap.get((v, u), 0) + 1; v = u
        flow += 1
    side = set(parent)
    return flow, [(u, v) for u, v, *_ in arrows if u in side and v not in side]

FLOW, MINCUT = max_flow(ARROWS, "MKT", "TRS")
print(f"most paths from the market to treasury sharing no arrow: {FLOW}; a cut of that size: "
      + ", ".join(f"{u}->{v}" for u, v in MINCUT))

def separates(removed, s="MKT", t="TRS"):
    return t not in bfs([a for a in ARROWS if (a[0], a[1]) not in removed], s)[0]

CASH = [("IND", "TRS"), ("DIG", "TRS"), ("MAT", "TRS")]
DEMAND = [("MKT", "IND"), ("MKT", "DIG"), ("MKT", "MAT")]
print(f"the three cash arrows into treasury separate it from the market: {separates(CASH)};"
      f" the three demand arrows out of the market: {separates(DEMAND)}")
SMALL = [c for r in (1, 2) for c in combinations(sorted(DECLARED), r) if separates(c)]
print(f"cuts of one or two arrows that separate them: {len(SMALL)} of the {14 + 91} tried")

## Panel 4 — Send a shock (AXIOM Lab 1.9, step 3; Theorem 9.2(ii)–(iii); Online Appendix 9.B.2; Section 9.8: routes 3, 10 and 40)

AXIOM Lab 1.9, step 3, the wavefront simulator and the path counter. Give every declared arrow a positive
sensitivity, drawn with the lab's seed, and shock the market: in this positive linear company each component first
moves exactly at its distance (Theorem 9.2(iii)), and what reaches treasury after two quarters is the sum, over the
three two-step routes, of each division's sensitivity to demand times the share of a change in its results that
reaches cash (Online Appendix 9.B.2). Then weaken the dependences. Halving a sensitivity changes how much arrives,
not when; switching arrows off can delay arrival or stop it, never bring it forward. In 10,000 situations drawn with
the lab's seed, each arrow switched off with probability 0.3, arrival comes later in many of them and earlier in none.
Cutting one or two cash arrows still leaves treasury exposed in the second quarter; cutting all three closes it off.
The path counter counts routes, which may revisit a component, from the market to treasury: 3 of two steps, 10 of
four and 40 of six (Section 9.8), and none of odd length, because among the components the market can reach every
arrow joins a division to one of the market, engineering and treasury (Online Appendix 9.B.7).

In [ ]:
def response(M, source="MKT", quarters=12):
    # the change in every component, quarter by quarter, after a unit shock to the source
    d = np.zeros(len(NODES)); d[IDX[source]] = 1.0; out = [d]
    for _ in range(quarters):
        d = M @ d; out.append(d)
    return np.array(out)

def first_move(R):
    return {v: next((k for k in range(len(R)) if R[k, IDX[v]] > 0), None) for v in NODES}

R = response(M)
FIRST = first_move(R)
print(f"a unit demand shock in the positive linear company (sensitivities drawn with seed {SEED}) first moves")
print("  " + "; ".join(f"{v} at quarter {FIRST[v]}" if FIRST[v] is not None else f"{v} never"
                      for v in ("IND", "DIG", "MAT", "ENG", "TRS", "GOV")))
print("  each exactly at its distance from the market:", all(FIRST[v] == DIST.get(v) for v in NODES))
ROUTE_SUM = sum(SENS[(dv, "TRS")] * SENS[("MKT", dv)] for dv in ("IND", "DIG", "MAT"))
print(f"what reaches treasury at quarter 2: {R[2, IDX['TRS']]:.6f}; the three two-step routes' products sum to {ROUTE_SUM:.6f}")

half = dict(SENS); half[("MKT", "IND")] /= 2
Rh = response(assemble(half))
print(f"halve Industrial Systems' sensitivity to demand: it still moves at quarter {first_move(Rh)['IND']}, by "
      f"{Rh[1, IDX['IND']] / R[1, IDX['IND']]:.2f} of the amount")
for label, off in (("IND->TRS off", CASH[:1]), ("IND->TRS and DIG->TRS off", CASH[:2]), ("all three cash arrows off", CASH)):
    f = first_move(response(assemble({a: (0.0 if a in off else s) for a, s in SENS.items()})))
    print(f"  {label:27s} " + (f"treasury first moves at quarter {f['TRS']}" if f["TRS"] is not None
                                 else "treasury never moves: the shock is cut off"))

# 10,000 situations (fresh draws, lab seed): positive sensitivities, each arrow switched off with probability 0.3
rng = np.random.default_rng(SEED)
N_SIT, P_OFF, QTRS = 10_000, 0.3, 12
on = rng.random((N_SIT, len(ARROWS))) >= P_OFF
S = rng.uniform(0.2, 0.8, (N_SIT, len(ARROWS))) * on
MS = np.zeros((N_SIT, len(NODES), len(NODES)))
MS[:, np.arange(len(NODES)), np.arange(len(NODES))] = rng.uniform(0.3, 0.9, (N_SIT, len(NODES)))
for j, (u, v, _) in enumerate(ARROWS):
    MS[:, IDX[v], IDX[u]] = S[:, j]
d = np.zeros((N_SIT, len(NODES))); d[:, IDX["MKT"]] = 1.0
MOVED = np.zeros((QTRS + 1, N_SIT, len(NODES)), dtype=bool); MOVED[0] = d > 0
for k in range(1, QTRS + 1):
    d = np.einsum("nij,nj->ni", MS, d); MOVED[k] = MOVED[k - 1] | (d > 0)
ARRIVE = np.where(MOVED.any(0), MOVED.argmax(0), -1)        # first quarter moved; -1: not within twelve
print(f"\n{N_SIT:,} situations (seed {SEED}), each arrow off with probability {P_OFF}:")
print(f"  {'component':9s} {'floor':>6s} {'earliest seen':>14s} {'at the floor':>13s} {'later':>7s} {'not within 12':>14s}")
EARLY = 0
for v in ("IND", "DIG", "MAT", "ENG", "TRS", "GOV"):
    a = ARRIVE[:, IDX[v]]; fl = DIST.get(v); hit = a[a >= 0]
    EARLY += int((hit < (fl if fl is not None else QTRS + 1)).sum())
    print(f"  {v:9s} {fl if fl is not None else 'never':>6} {(hit.min() if len(hit) else '-'):>14} "
          f"{(a == fl).mean():13.1%} {((a > (fl or 0)) & (a >= 0)).mean():7.1%} {(a < 0).mean():14.1%}")
print(f"situations in which a component moved before its distance: {EARLY}")

WALKS = {k: int(np.linalg.matrix_power(A, k)[IDX["MKT"], IDX["TRS"]]) for k in range(1, 9)}
print("\nroutes (walks) from the market to treasury, by length:", ", ".join(f"{k}: {w}" for k, w in WALKS.items()))
reach = [v for v in NODES if v in DIST]
colour, stack = {"MKT": 0}, ["MKT"]                     # two-colour what the market can reach
while stack:
    u = stack.pop()
    for a, b, _ in ARROWS:
        for x, y in ((a, b), (b, a)):
            if x == u and y in reach and y not in colour:
                colour[y] = 1 - colour[u]; stack.append(y)
SPLIT = all(colour[a] != colour[b] for a, b, _ in ARROWS if a in reach and b in reach)
print(f"every arrow among {{{', '.join(v for v in reach if colour[v] == 0)}}} and {{{', '.join(v for v in reach if colour[v] == 1)}}}"
      f" joins the two groups: {SPLIT}, so every route from the market to treasury has even length")

ROWS = ["IND", "DIG", "MAT", "ENG", "TRS", "GOV"]
KS = np.arange(7)
SHARE = np.array([[MOVED[k, :, IDX[v]].mean() for k in KS] for v in ROWS])
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.6, 3.6), gridspec_kw={"width_ratios": [1.6, 1]})
ax.imshow(SHARE, cmap="Blues", vmin=0, vmax=1.15, aspect="auto")
for i, v in enumerate(ROWS):
    for k in KS:
        ax.text(k, i, f"{100 * SHARE[i, k]:.0f}%", ha="center", va="center", fontsize=6.5,
                color="white" if SHARE[i, k] > 0.75 else "#1A1A1A")
xs, ys = [], []                                   # the floor as a staircase: left of it, no situation ever moves
for i, v in enumerate(ROWS):
    x = DIST[v] - 0.5 if v in DIST else KS[-1] + 0.5
    xs += [x, x]; ys += [i - 0.5, i + 0.5]
ax.plot(xs, ys, color="#B71C1C", lw=2.2, label="the floor: each component's distance from the market")
ax.set_xticks(KS); ax.set_xlim(-0.5, KS[-1] + 0.5)
ax.set_yticks(range(len(ROWS)), [f"{v} ({'never' if v not in DIST else 'floor ' + str(DIST[v])})" for v in ROWS], fontsize=7.5)
ax.set_xlabel("quarters after the demand shock", fontsize=8)
ax.set_title("share of 10,000 situations in which each has moved", fontsize=8.5)
ax.legend(frameon=False, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.13))
ax.spines[:].set_visible(False); ax.tick_params(labelsize=7.5, length=0)
L = np.array(list(WALKS)); W = np.array(list(WALKS.values()))
bx.bar(L[W > 0], W[W > 0], color="#1F4E79", width=0.62)
for k, w in zip(L, W):
    bx.text(k, max(w, 1) * 1.18, str(w), ha="center", va="bottom", fontsize=7.5, color="#1A1A1A" if w else "#999999")
bx.set_yscale("log"); bx.set_ylim(0.8, 700); bx.set_xlim(0.3, 8.7); bx.set_xticks(L)
bx.set_yticks([1, 10, 100], ["1", "10", "100"]); bx.minorticks_off()
bx.set_xlabel("route length (quarters)", fontsize=8); bx.set_ylabel("routes, market to treasury", fontsize=8)
bx.set_title("the path counter", fontsize=8.5); bx.tick_params(labelsize=7.5)
fig.suptitle("No component moves before its distance from the shock (seed 26109)", fontsize=10)
plt.tight_layout(); plt.show()

## Panel 5 — Test the summary map (Worked Example 9.1, Step 5; Theorem 9.3; Figure 9.2; Online Appendix 9.B.3)

Worked Example 9.1, Step 5, and AXIOM Lab 1.9, step 4, the aggregation inspector. On the two-box map the board usually
sees, the operating divisions and the corporate center (engineering, treasury and governance), the center reaches
the divisions. Drill down: that is true for Industrial Systems and Digital and false for Materials, which nothing in
the center reaches, so a claim such as "a liquidity squeeze at the center will hit Materials" cannot be supported
from the summary map, and on the declared wiring it is false. The same drill-down finds the summary overstating the
reach into the center as well: governance, a driver, is reached by nothing. Aggregation never loses a real path or
lengthens a distance (Theorem 9.3(i)), but it can invent connections (Theorem 9.3(ii)), here because the divisions
box is not strongly connected: no arrow runs between two divisions. So the absence of a path on a summary map proves
a separation; its presence proves nothing. Aggregate instead by causal structure, into the market, governance,
Materials and the core, each strongly connected within itself, and the summary map invents no connection (Online
Appendix 9.B.3). The missing arrows are claims as well: no arrow points into the market, so the architecture asserts
that nothing Meridian does moves its demand within the planning horizon. In the figure, each number is the quarters a
shock needs to travel from one component to another, a blank means no path, and a cross marks a pair that the
summary map shows as connected and the wiring does not.

In [ ]:
def aggregate(arrows, part):
    # Definition 9.6: aggregates as nodes; an arrow A -> B (A != B) whenever some arrow runs from A to B
    return sorted({(part[u], part[v]) for u, v, *_ in arrows if part[u] != part[v]})

def summary_test(part):
    # compare every coarse connection with the components it claims to connect
    boxes = list(dict.fromkeys(part[v] for v in NODES))
    coarse = aggregate(ARROWS, part)
    cdist = {a: bfs(coarse, a, boxes)[0] for a in boxes}
    fine = {u: bfs(ARROWS, u)[0] for u in NODES}
    invented = [(u, v) for u in NODES for v in NODES
                if part[u] != part[v] and part[v] in cdist[part[u]] and v not in fine[u]]
    lost = [(u, v) for u in NODES for v in fine[u]
            if part[u] != part[v] and cdist[part[u]].get(part[v], 99) > fine[u][v]]
    return boxes, coarse, cdist, fine, invented, lost

BOXES, COARSE, CDIST, FINE, INVENTED, LOST = summary_test(TWO_BOX)
print("two-box map:", ", ".join(f"{a} -> {b}" for a, b in COARSE))
print(f"on the two-box map the center reaches the divisions: {'divisions' in CDIST['center']}")
CENTER = [v for v in BOOK_ORDER if TWO_BOX[v] == "center"]
DIVS = [v for v in BOOK_ORDER if TWO_BOX[v] == "divisions"]
print("drill down, distance from each member of the center to each division:")
print("        " + "".join(f"{v:>7s}" for v in DIVS))
for u in CENTER:
    print(f"  {u:5s} " + "".join(f"{FINE[u].get(v, 'none'):>7}" for v in DIVS))
REACHED = [v for v in DIVS if any(v in FINE[u] for u in CENTER)]
TO_MAT = sum("MAT" in FINE[u] for u in CENTER)
print(f"divisions the center really reaches: {', '.join(REACHED)}; members of the center that reach Materials: {TO_MAT}")
print(f"a liquidity squeeze at treasury reaches Materials: {'MAT' in FINE['TRS']}")
print("connections the two-box map shows and the wiring lacks:", ", ".join(f"{u}->{v}" for u, v in INVENTED))
print(f"real paths lost or lengthened by aggregation: {len(LOST)} (Theorem 9.3(i))")
INNER = [a for a in ARROWS if a[0] in DIVS and a[1] in DIVS]
print(f"arrows between two divisions: {len(INNER)}, so the divisions box is strongly connected within itself:"
      f" {len(tarjan(INNER, DIVS)) == 1}")

LAYERED = {v: ("core" if v in CORE else NAME[v].lower()) for v in NODES}
_, COARSE_L, _, _, INVENTED_L, LOST_L = summary_test(LAYERED)
print("\nthe causal-layer map:", ", ".join(f"{a} -> {b}" for a, b in COARSE_L))
print(f"connections it invents: {len(INVENTED_L)}; real paths it loses: {len(LOST_L)} (Online Appendix 9.B.3)")

# Theorem 9.3(ii): components u, a1, a2, w with arrows u -> a1 and a2 -> w; merge a1 and a2 into A
TOY = [("u", "a1"), ("a2", "w")]
toy_part = {"u": "u", "a1": "A", "a2": "A", "w": "w"}
TOY_COARSE = "w" in bfs(aggregate(TOY, toy_part), "u", ["u", "A", "w"])[0]
TOY_FINE = "w" in bfs(TOY, "u", ["u", "a1", "a2", "w"])[0]
print(f"Theorem 9.3(ii): the merged map has a path from u to w: {TOY_COARSE}; the components do: {TOY_FINE}")
print(f"Proposition 9.4(c): the core meets both boxes of the two-box map ({' and '.join(by_box['divisions'])} in the"
      f" divisions, {' and '.join(by_box['center'])} in the center), so no level of that hierarchy is a union of causal layers")
print(f"arrows into the market: {int(A[:, IDX['MKT']].sum())}; into governance: {int(A[:, IDX['GOV']].sum())}")

fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.6))
for ax, part, inv, head, order in (
        (axes[0], TWO_BOX, INVENTED, "two-box map: market | divisions | center", ["MKT", "IND", "DIG", "MAT", "ENG", "TRS", "GOV"]),
        (axes[1], LAYERED, INVENTED_L, "causal map: market | governance | Materials | core", ["MKT", "GOV", "MAT", "IND", "DIG", "ENG", "TRS"])):
    boxes = list(dict.fromkeys(part[v] for v in order))
    filled = np.array([[1.0 if (v != u and v in FINE[u]) else 0.0 for v in order] for u in order])
    ax.imshow(filled, cmap="Blues", vmin=0, vmax=2.2)
    for i, u in enumerate(order):
        for j, v in enumerate(order):
            if v != u and v in FINE[u]:
                ax.text(j, i, str(FINE[u][v]), ha="center", va="center", fontsize=7.5, color="#1A1A1A")
            elif (u, v) in inv:
                ax.add_patch(mp.Rectangle((j - 0.5, i - 0.5), 1, 1, fc="#F6D5D5", ec="#B71C1C", lw=1.0))
                ax.text(j, i, "×", ha="center", va="center", fontsize=9, color="#B71C1C")
    edges = np.cumsum([sum(part[v] == b for v in order) for b in boxes])[:-1] - 0.5
    for e in edges:
        ax.axhline(e, color="#616161", lw=0.8); ax.axvline(e, color="#616161", lw=0.8)
    ax.set_xticks(range(len(order)), order, fontsize=7, rotation=90); ax.set_yticks(range(len(order)), order, fontsize=7)
    ax.set_xlabel("to (number: quarters to arrive)", fontsize=8); ax.set_ylabel("from", fontsize=8)
    ax.set_title(f"{head}\n" + (f"{len(inv)} connections invented (×)" if inv else "no connection invented"), fontsize=8)
    ax.tick_params(length=0); ax.spines[:].set_visible(False)
fig.suptitle("Summary maps prove separations, never connections (seed 26109)", fontsize=10)
plt.tight_layout(); plt.show()

## Panel 6 — Audit the constraints (Worked Example 9.1, Step 4; Theorem 9.4; Figure 9.3; Online Appendix 9.B.4–9.B.6: β = 1, one joint review, risk peaks at 73.6)

AXIOM Lab 1.9, step 5, the consistency bench, and Worked Example 9.1, Step 4. First the three planning rules of
Theorem 9.4(i): the budget at least a month after the capacity plan, the capacity plan at least a month after the
hiring plan, and the hiring plan at least a month after the budget. Each passes a bilateral check and any two can be
met, but together they would put the budget three months after itself: the rules form a cycle of total weight
$-3$, and dates under such rules are feasible exactly when no cycle has negative weight (Online Appendix 9.B.4). The
same test finds the carve-out timetable of Online Appendix 9.B.5 a month short, and feasible, with the milestones a
month apart, once the buyer's window is three months. Then Meridian's constraints: the constraint graph joins IND,
DIG and MAT in a triangle, through the risk ceiling, and hangs engineering off Digital, through the capability floor;
its cycle measure is $\beta = 4 - 7 + 4 = 1$ (four links, seven components, four separate pieces). One cycle means
exactly one joint review: the three division heads review the risk ceiling together; a bilateral check of the
capability floor against the Digital plans that pass that review completes the certificate (Online Appendix 9.B.6),
and every other limit is checked by its owner alone. The joint review is not a formality: on the digital plan of
Chapter 1 (Equation 1.B.1, Table 1.B.1), group strategic risk peaks at 73.6 about eleven months in, 1.4 points under
the ceiling and well inside the risk committee's measurement error of ±6 points (Table 5.2). Fresh draws with the
lab's seed show that the triangle's failure is no curiosity: when each of three constraints matches every setting of
one unit with exactly one setting of the other, every bilateral check passes, yet a third of the triangles cannot be
met together; on a tree (Theorem 9.4(ii)) none fails.

In [ ]:
def feasible(n_dates, rules):
    # Online Appendix 9.B.4: dates with x_j <= x_i + b for every rule (i, j, b); least walk weights by n rounds of
    # relaxation; any rule still violated means a cycle of negative weight, and no plan exists
    x = np.zeros(n_dates)
    for _ in range(n_dates):
        for i, j, b in rules:
            x[j] = min(x[j], x[i] + b)
    return all(x[j] <= x[i] + b + 1e-12 for i, j, b in rules), x

# Theorem 9.4(i): budget (0), capacity plan (1), hiring plan (2); "Y at least a month after X" is x_X <= x_Y - 1
PLAN = [(0, 1, -1.0),      # the budget at least a month after the capacity plan
        (1, 2, -1.0),      # the capacity plan at least a month after the hiring plan
        (2, 0, -1.0)]      # the hiring plan at least a month after the budget
print("bilateral checks: each rule matches any date of either unit with a date of the other (arc-consistent on the line)")
PAIRS_OK = sum(feasible(3, list(pair))[0] for pair in combinations(PLAN, 2))
EX = np.array([2.0, 1.0, 0.0])                       # budget, capacity, hiring: the first two rules hold
print(f"pairs of rules that can be met together: {PAIRS_OK} of 3; e.g. budget 2, capacity 1, hiring 0 meets the first"
      f" two: {all(EX[j] <= EX[i] + b for i, j, b in PLAN[:2])}, not the third: {EX[0] <= EX[2] - 1}")
PLAN_OK, _ = feasible(3, PLAN)
PLAN_W = sum(b for *_, b in PLAN)
print(f"all three together: {'feasible' if PLAN_OK else 'infeasible'}; the cycle's weight is {PLAN_W:+.0f}:"
      f" together the rules put the budget {-PLAN_W:.0f} months after itself")

# Online Appendix 9.B.5: signing (0), approval (1), separation (2), closing (3), months
def carve_out(window):
    return [(1, 0, -1.0), (2, 1, -1.0), (3, 2, -1.0), (0, 3, float(window))]
CO_OK, _ = feasible(4, carve_out(2))
CO_THREE = sum(feasible(4, list(c))[0] for c in combinations(carve_out(2), 3))
CO_W = 2 - 1 - 1 - 1
CO3_OK, CO3_X = feasible(4, carve_out(3))
CO3_M = CO3_X - CO3_X[0]
print(f"\ncarve-out, closing within two months of signing: {'feasible' if CO_OK else 'infeasible'}; cycle weight {CO_W:+.0f};"
      f" any three of the four constraints: {CO_THREE} of 4 sets feasible")
print(f"with a three-month window: {'feasible' if CO3_OK else 'infeasible'}, milestones at months {CO3_M.round(0).astype(int).tolist()}")

# Meridian's constraint graph: each coupling joins every pair in its scope (Definition 9.7)
CLINKS_C = sorted({tuple(sorted(p, key=BOOK_ORDER.index)) for _, scope, _ in CONSTRAINTS for p in combinations(scope, 2)},
                  key=lambda e: (BOOK_ORDER.index(e[0]), BOOK_ORDER.index(e[1])))
CPIECES = pieces(CLINKS_C, BOOK_ORDER)
C_BETA = len(CLINKS_C) - len(NODES) + len(CPIECES)
print(f"\nconstraint graph: {len(CLINKS_C)} links ({', '.join(f'{a}-{b}' for a, b in CLINKS_C)}); separate pieces "
      + "  ".join("{" + ", ".join(p) + "}" for p in CPIECES))
print(f"cycle measure beta = {len(CLINKS_C)} - {len(NODES)} + {len(CPIECES)} = {C_BETA}")

def fundamental_cycles(links, nodes=NODES):
    # a spanning forest by breadth-first search; each link outside it closes one cycle (Proposition 9.6)
    nbr = {v: [] for v in nodes}
    for a, b in links:
        nbr[a].append(b); nbr[b].append(a)
    parent = {}
    for s in nodes:
        if s not in parent:
            parent[s], queue = None, [s]
            while queue:
                a = queue.pop(0)
                for b in sorted(nbr[a], key=nodes.index):
                    if b not in parent:
                        parent[b] = a; queue.append(b)
    tree = {frozenset((v, p)) for v, p in parent.items() if p is not None}
    def up(v):
        path = [v]
        while parent[path[-1]] is not None:
            path.append(parent[path[-1]])
        return path
    cycles = []
    for a, b in links:
        if frozenset((a, b)) not in tree:
            pa, pb = up(a), up(b)
            top = next(x for x in pa if x in pb)
            cycles.append(pa[:pa.index(top) + 1] + pb[:pb.index(top)][::-1])
    return cycles

REVIEWS = fundamental_cycles(CLINKS_C)
print(f"joint reviews, one per cycle, of exactly its members: {len(REVIEWS)}: " + "; ".join("{" + ", ".join(sorted(c, key=BOOK_ORDER.index)) + "}" for c in REVIEWS))
BLOCK = {v: ("joint review" if any(v in c for c in REVIEWS) else v) for v in NODES}
GSTAR = [(BLOCK[a], BLOCK[b]) for a, b in CLINKS_C if BLOCK[a] != BLOCK[b]]
blocks = list(dict.fromkeys(BLOCK[v] for v in BOOK_ORDER))
GSTAR_BETA = len(GSTAR) - len(blocks) + len(pieces(GSTAR, blocks))
print(f"Online Appendix 9.B.6: blocks {', '.join(blocks)}; links between blocks: {', '.join(f'{a} - {b}' for a, b in GSTAR)};"
      f" cycle measure {GSTAR_BETA}: a forest, so one bilateral check of the capability floor completes the certificate")
print(f"local constraints, checked by their owners alone: {sum(len(s) == 1 for _, s, _ in CONSTRAINTS)} (treasury: 2, Industrial Systems: 1)")

# fresh draws (lab seed): three units with three settings each; each constraint matches every setting of one unit
# with exactly one setting of the other, so it passes its bilateral check by construction
rng = np.random.default_rng(SEED)
N_FAM = 20_000
MATCH = rng.permuted(np.tile(np.arange(3), (N_FAM, 3, 1)), axis=2)      # R0(x, y), R1(y, z), R2(z, x)
REL = np.zeros((N_FAM, 3, 3, 3), dtype=int)
REL[np.arange(N_FAM)[:, None, None], np.arange(3)[None, :, None], np.arange(3)[None, None, :], MATCH] = 1
BILATERAL = int(((REL.max(axis=3) == 1).all(axis=2) & (REL.max(axis=2) == 1).all(axis=2)).all(axis=1).sum())
TRI = np.einsum("nxy,nyz,nzx->n", REL[:, 0], REL[:, 1], REL[:, 2]) > 0
TREE = np.einsum("nxy,nyz->n", REL[:, 0], REL[:, 1]) > 0
TRI_FAIL, TREE_FAIL = int((~TRI).sum()), int((~TREE).sum())
print(f"\n{N_FAM:,} random families of matching constraints (seed {SEED}), {BILATERAL:,} passing every bilateral check:"
      f" on a triangle {TRI_FAIL:,} ({TRI_FAIL / N_FAM:.1%}) cannot be met together (a third in expectation: around the"
      f" cycle the matchings compose to a rearrangement of the three settings, and 2 of the 6 leave none in place);"
      f" on a tree, two of the three links, {TREE_FAIL}")

def strategic_risk(t):
    # Chapter 1's digital plan (Equation 1.B.1, Table 1.B.1): from 64 toward 51 at speed 0.40, with a trough of
    # +14 reached at 1.2 years (t in years)
    return 51 + (64 - 51) * np.exp(-0.40 * t) + 14 * (t / 1.2) * np.exp(1 - t / 1.2)
pk = minimize_scalar(lambda t: -strategic_risk(t), bounds=(0.2, 2.0), method="bounded", options={"xatol": 1e-10})
PEAK, PEAK_MONTH = float(-pk.fun), float(12 * pk.x)
MONTHS = np.arange(61)
RISK_M = strategic_risk(MONTHS / 12)
GRID_MONTH = int(RISK_M.argmax())
GAP = CEILING - PEAK
print(f"\ndigital plan: group strategic risk peaks at {PEAK:.2f} at month {PEAK_MONTH:.1f} (monthly grid: {RISK_M.max():.2f}"
      f" in month {GRID_MONTH}), {GAP:.2f} points under the ceiling of {CEILING:.0f}, inside the +/-{RISK_ERROR:.0f} error: {GAP < RISK_ERROR}")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
m = MONTHS[:37]
ax.fill_between(m, RISK_M[:37] - RISK_ERROR, RISK_M[:37] + RISK_ERROR, color="#1F4E79", alpha=0.12, lw=0,
                label="the risk committee's measurement error, ±6")
ax.plot(m, RISK_M[:37], color="#1F4E79", lw=2, label="group strategic risk on the digital plan")
ax.axhline(CEILING, color="#B71C1C", lw=1.2, ls="--", label="the board's ceiling, 75")
ax.plot([PEAK_MONTH], [PEAK], "o", color="#E65100", ms=5)
ax.annotate(f"peak {PEAK:.1f} at month {PEAK_MONTH:.0f}:\n{GAP:.1f} points under the ceiling", xy=(PEAK_MONTH, PEAK),
            xytext=(17, 63), fontsize=8, color="#E65100", arrowprops=dict(arrowstyle="-", color="#E65100", lw=0.7))
ax.set(xlabel="months from today", ylabel="strategic risk (index, 0-100)", xlim=(0, 36), ylim=(50, 82),
       xticks=range(0, 37, 6), title="On the digital plan, risk peaks 1.4 points under the ceiling (seed 26109)")
ax.legend(frameon=False, fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

## Panel 7 — Run the validation checklist and read the metrics (Tables 9.4–9.5; Theorem 9.5; Section 9.8; Online Appendix 9.B.7: 14 arrows, density 0.33, β = 8)

AXIOM Lab 1.9, step 5, second part: run the validation checklist row by row (Table 9.4), each row a checkable claim,
and read the complexity panel (Definition 9.10; Table 9.5). Meridian's dependency graph has 7 components and 14
arrows, a density of $14/42 = 0.33$, and a cycle measure $\beta = 14 - 7 + 1 = 8$, each two-way relationship counting
as a loop: the four two-way relationships contribute four loops, and the remaining four come from the graph with
directions ignored. It has one feedback core of four members containing six loops, and a condensation of depth 2;
its constraint graph is much sparser, with $\beta = 1$. Completeness is a coverage check (Theorem 9.5): the
architecture can bound how soon a demand shock reaches the covenant, because that question's support (the market, the
divisions, treasury and the arrows among them) is declared, but it cannot say which customers' orders will fall
first, because no component holds customer-level readings. None of these numbers says how strong the loops are:
the metrics read the wiring, and pricing what they find is the work of Part II's dynamic instruments.

In [ ]:
def panel(arrows):
    # Definition 9.10: the complexity panel of a dependency graph
    n_, m_ = len(NODES), len(arrows)
    p_ = len(pieces(arrows))
    pairs = {(u, v) for u, v, *_ in arrows}
    two = sum((v, u) in pairs for u, v in pairs) // 2
    comps, _, _, level = condensation(arrows)
    cores = [c for c in comps if len(c) > 1]
    return {"components": n_, "arrows": m_, "density": m_ / (n_ * (n_ - 1)), "pieces": p_, "beta": m_ - n_ + p_,
            "two_way": two, "beta_undirected": (m_ - two) - n_ + p_, "cores": len(cores),
            "core_sizes": [len(c) for c in cores], "loops": len(simple_cycles(arrows)),
            "layers": max(level.values()), "depth": max(level.values()) - 1}

PANEL = panel(ARROWS)
print("the complexity panel (Definition 9.10):")
print(f"  size: {PANEL['components']} components, {PANEL['arrows']} arrows; density {PANEL['arrows']}/{PANEL['components'] * (PANEL['components'] - 1)}"
      f" = {PANEL['density']:.3f}")
print(f"  cycle measure: beta = {PANEL['arrows']} - {PANEL['components']} + {PANEL['pieces']} = {PANEL['beta']}"
      f" = {PANEL['two_way']} two-way relationships + {PANEL['beta_undirected']} loops with directions ignored")
print(f"  feedback mass: {PANEL['cores']} core of {PANEL['core_sizes'][0]} members, {PANEL['loops']} directed loops;"
      f" condensation depth {PANEL['depth']}")
print(f"  distance profile from the market: " + ", ".join(f"{v} {DIST.get(v, 'never')}" for v in ("IND", "DIG", "MAT", "ENG", "TRS", "GOV")))
print(f"  constraint graph: {len(CLINKS_C)} links, cycle measure {C_BETA}")
TABLE95 = [("Density", "overall intensity of coupling", "where the coupling sits"),
           ("Cycle measure", "independent loops; the joint-review burden", "how strong the loops are, and their signs"),
           ("Feedback mass", "extent of the mutual-influence cores", "the gains inside the cores"),
           ("Condensation depth", "the longest one-way causal chain", "actual propagation speeds"),
           ("Distance profile", "floors on shock arrival", "how much arrives")]
print("\nTable 9.5: what each measure reads, and what it cannot see")
for name, reads, blind in TABLE95:
    print(f"  {name:19s} reads {reads:44s} cannot see {blind}")

# Table 9.4, row by row, with the evidence computed above
COVENANT_SUPPORT = {"components": {"MKT", "IND", "DIG", "MAT", "TRS"},
                    "arrows": {("MKT", d) for d in ("IND", "DIG", "MAT")} | {(d, "TRS") for d in ("IND", "DIG", "MAT")}}
COVERED = COVENANT_SUPPORT["components"] <= set(NODES) and COVENANT_SUPPORT["arrows"] <= DECLARED
CHECKLIST = [
    ("Interfaces", f"{BACKED} of {len(ARROWS)} arrows interface-backed: the components assemble"),
    ("Wiring", f"{len(CANON)} of {len(ARROWS)} arrows witnessed active (Panel 1); {MISSING} missing arrows listed as claims;"
               f" {len(CORES)} core, {LAYERS} layers"),
    ("Constraints", f"{len(CONSTRAINTS)} scopes declared; {len(REVIEWS)} cycle listed; bilateral sign-off on the tree-shaped"
                    f" rest and {len(REVIEWS)} joint review"),
    ("Hierarchy", f"the two-box map used one way only: it invents {len(INVENTED)} connections, the causal-layer map"
                  f" {len(INVENTED_L)}"),
    ("Completeness", f"the covenant question's support covered: {COVERED}; which customers' orders fall first: not"
                     f" answerable without customer-level readings"),
]
print("\nTable 9.4, the validation checklist, on Meridian's architecture")
for stage, evidence in CHECKLIST:
    print(f"  {stage:13s} {evidence}")

## Panel 8 — Rewire: treasury funds Materials (Worked Example 9.1, For discussion; Proposition 9.6; Online Appendix 9.B.7)

The question the worked example leaves for discussion, and the lab's second discussion question: suppose the board
decides that treasury will fund Materials' next capacity expansion. The arrow added is funding, TRS → MAT. Materials
joins the core, which grows to five members; the layers collapse to drivers and core, with condensation depth 1; the
two-box map's connection from the center to Materials becomes real; and the constraint graph, and with it the single
joint review, is unchanged (Online Appendix 9.B.7). As Proposition 9.6 says of any added arrow, the density rises by
1/42, the cycle measure by one (treasury and Materials already lie in one piece), routes only multiply, and no
arrival floor lengthens; the market reached Materials in one quarter already, so the floors from the market stay
where they were.

In [ ]:
WHATIF = ARROWS + [("TRS", "MAT", "funding")]
PANEL_W = panel(WHATIF)
COMPS_W, _, _, LEVEL_W = condensation(WHATIF)
CORE_W = [c for c in COMPS_W if len(c) > 1][0]
print(f"the new arrow: TRS -> MAT (funding); feedback core {{{', '.join(CORE_W)}}}, {len(CORE_W)} members")
for lv in sorted(set(LEVEL_W.values())):
    print(f"  layer {lv}: " + "  ".join("{" + ", ".join(COMPS_W[i]) + "}" for i in LEVEL_W if LEVEL_W[i] == lv))
FINE_W = {u: bfs(WHATIF, u)[0] for u in NODES}
CENTER_MAT_W = sum("MAT" in FINE_W[u] for u in CENTER)
print(f"members of the center that now reach Materials: {CENTER_MAT_W} (" + ", ".join(f"{u} in {FINE_W[u]['MAT']}" for u in CENTER) + ")")
DIST_W = bfs(WHATIF, "MKT")[0]
print("arrival floors from the market:", ", ".join(f"{v} {DIST_W.get(v, 'never')}" for v in ("IND", "DIG", "MAT", "ENG", "TRS", "GOV")),
      f"(unchanged: {DIST_W == DIST})")
WALKS_W = {k: int(np.linalg.matrix_power(adjacency(WHATIF), k)[IDX["MKT"], IDX["TRS"]]) for k in range(1, 7)}
print("routes from the market to treasury by length:", ", ".join(f"{k}: {WALKS[k]} -> {WALKS_W[k]}" for k in range(1, 7)))
print(f"\n{'':22s} {'today':>8s} {'TRS funds MAT':>14s}")
for key, label, fmt in (("arrows", "arrows", "{:.0f}"), ("density", "density", "{:.3f}"), ("beta", "cycle measure", "{:.0f}"),
                        ("loops", "directed loops", "{:.0f}"), ("layers", "causal layers", "{:.0f}"), ("depth", "condensation depth", "{:.0f}")):
    print(f"{label:22s} {fmt.format(PANEL[key]):>8s} {fmt.format(PANEL_W[key]):>14s}")
print(f"{'core members':22s} {PANEL['core_sizes'][0]:>8d} {len(CORE_W):>14d}")
print(f"{'joint reviews':22s} {len(REVIEWS):>8d} {len(fundamental_cycles(CLINKS_C)):>14d}   (the constraints are unchanged)")

## Validation — the book's numbers, and agreement with `DCT_V1_Ch09_Lab.xlsx`

Each checkpoint is a number printed in Chapter 9 (Worked Example 9.1, Table 9.7, Figures 9.1–9.3, Theorem 9.4,
Section 9.8, LOS Guidance) or in Online Appendix 9.B (9.B.1, 9.B.5–9.B.7); the strategic-risk peak is Chapter 1's
digital plan, which Step 4 cites. Three checkpoints marked *seed 26109* restate a theorem on the fresh draws. The
workbook recomputes the ones marked *workbook* with spreadsheet formulas and shows PASS against the same engine values
on its `Checks` sheet.

In [ ]:
D_ = lambda v: DIST.get(v, -1)
checks = [
    ("components (workbook)", len(NODES), 7, 0),
    ("typed arrows (workbook)", len(ARROWS), 14, 0),
    ("demand arrows (workbook)", sum(t == "demand" for *_, t in ARROWS), 3, 0),
    ("cash arrows (workbook)", sum(t == "cash" for *_, t in ARROWS), 3, 0),
    ("funding arrows (workbook)", sum(t == "funding" for *_, t in ARROWS), 2, 0),
    ("capability arrows (workbook)", sum(t == "capability" for *_, t in ARROWS), 4, 0),
    ("policy arrows (workbook)", sum(t == "policy" for *_, t in ARROWS), 2, 0),
    ("funding arrows into Materials (workbook)", FUND_MAT, 0, 0),
    ("arrows into the market (workbook)", int(A[:, IDX["MKT"]].sum()), 0, 0),
    ("possible arrows |V|(|V| - 1) (workbook)", n * (n - 1), 42, 0),
    ("density (workbook)", PANEL["density"], 0.333, 5e-4),
    ("cycle measure of the dependency graph (workbook)", PANEL["beta"], 8, 0),
    ("two-way relationships (workbook)", PANEL["two_way"], 4, 0),
    ("loops with directions ignored (workbook)", PANEL["beta_undirected"], 4, 0),
    ("feedback cores (workbook)", len(CORES), 1, 0),
    ("members of the core (workbook)", len(CORE), 4, 0),
    ("directed loops in the core (workbook)", len(LOOPS), 6, 0),
    ("loops through all four members (workbook)", len(LOOPS) - len(TWO_WAY), 2, 0),
    ("causal layers (workbook)", LAYERS, 3, 0),
    ("layer of Materials, the feeder (workbook)", LEVEL[OF["MAT"]], 2, 0),
    ("condensation depth (workbook)", DEPTH, 2, 0),
    ("earliest quarter: Industrial Systems (workbook)", D_("IND"), 1, 0),
    ("earliest quarter: Digital (workbook)", D_("DIG"), 1, 0),
    ("earliest quarter: Materials (workbook)", D_("MAT"), 1, 0),
    ("earliest quarter: engineering (workbook)", D_("ENG"), 2, 0),
    ("earliest quarter: treasury, the covenant (workbook)", D_("TRS"), 2, 0),
    ("governance reached by the shock (workbook)", int("GOV" in DIST), 0, 0),
    ("reached after one quarter (workbook)", sum(1 for v in DIST if 0 < DIST[v] <= 1), 3, 0),
    ("reached after two quarters (workbook)", sum(1 for v in DIST if 0 < DIST[v] <= 2), 5, 0),
    ("paths from the market to treasury (workbook)", len(PATHS), 5, 0),
    ("paths of two steps (workbook)", sum(len(p) == 3 for p in PATHS), 3, 0),
    ("paths of four steps (workbook)", sum(len(p) == 5 for p in PATHS), 2, 0),
    ("most paths sharing no arrow (workbook)", FLOW, 3, 0),
    ("fewest arrows to cut (workbook)", len(MINCUT), 3, 0),
    ("cuts of one or two arrows that separate (workbook)", len(SMALL), 0, 0),
    ("three cash arrows separate treasury (workbook)", int(separates(CASH)), 1, 0),
    ("three demand arrows separate treasury (workbook)", int(separates(DEMAND)), 1, 0),
    ("routes of two steps, market to treasury (workbook)", WALKS[2], 3, 0),
    ("routes of four steps (workbook)", WALKS[4], 10, 0),
    ("routes of six steps (workbook)", WALKS[6], 40, 0),
    ("routes of odd length up to seven (workbook)", WALKS[1] + WALKS[3] + WALKS[5] + WALKS[7], 0, 0),
    ("first move at treasury, positive company (seed 26109, workbook)", FIRST["TRS"], 2, 0),
    ("moves before the distance in 10,000 situations (seed 26109)", EARLY, 0, 0),
    ("two-box map: the center reaches the divisions (workbook)", int("divisions" in CDIST["center"]), 1, 0),
    ("divisions the center really reaches (workbook)", len(REACHED), 2, 0),
    ("members of the center that reach Materials (workbook)", TO_MAT, 0, 0),
    ("connections the causal-layer map invents (workbook)", len(INVENTED_L), 0, 0),
    ("Theorem 9.3(ii): merged map has the path", int(TOY_COARSE and not TOY_FINE), 1, 0),
    ("OA 9.B.1: consolidated profit moved by the rule (workbook)", OFF_MOVE, 0, 0),
    ("OA 9.B.1: next rule's rise per unit of profit (workbook)", OFF_BACK, 0.5, 1e-9),
    ("planning rules: pairs that can be met (workbook)", PAIRS_OK, 3, 0),
    ("planning rules: all three feasible (workbook)", int(PLAN_OK), 0, 0),
    ("planning rules: cycle weight, months (workbook)", PLAN_W, -3, 0),
    ("OA 9.B.5: carve-out cycle weight (workbook)", CO_W, -1, 0),
    ("OA 9.B.5: carve-out feasible in two months (workbook)", int(CO_OK), 0, 0),
    ("OA 9.B.5: sets of three that can be met (workbook)", CO_THREE, 4, 0),
    ("OA 9.B.5: feasible with a three-month window (workbook)", int(CO3_OK), 1, 0),
    ("OA 9.B.5: months between milestones (workbook)", float(np.diff(CO3_M).mean()), 1, 1e-9),
    ("constraint graph: links (workbook)", len(CLINKS_C), 4, 0),
    ("constraint graph: separate pieces (workbook)", len(CPIECES), 4, 0),
    ("constraint graph: cycle measure (workbook)", C_BETA, 1, 0),
    ("joint reviews (workbook)", len(REVIEWS), 1, 0),
    ("members of the joint review (workbook)", len(REVIEWS[0]), 3, 0),
    ("OA 9.B.6: links between blocks (workbook)", len(GSTAR), 1, 0),
    ("OA 9.B.6: cycle measure of the blocks (workbook)", GSTAR_BETA, 0, 0),
    ("arc-consistent tree families that fail (seed 26109)", TREE_FAIL, 0, 0),
    ("strategic-risk peak on the digital plan (workbook)", PEAK, 73.6, 0.05),
    ("its month, about eleven (workbook)", PEAK_MONTH, 11, 0.5),
    ("monthly grid: month of the peak (workbook)", GRID_MONTH, 11, 0),
    ("points under the ceiling of 75 (workbook)", GAP, 1.4, 0.05),
    ("inside the +/-6 measurement error (workbook)", int(GAP < RISK_ERROR), 1, 0),
    ("what-if TRS->MAT: members of the core (workbook)", len(CORE_W), 5, 0),
    ("what-if: causal layers (workbook)", PANEL_W["layers"], 2, 0),
    ("what-if: condensation depth (workbook)", PANEL_W["depth"], 1, 0),
    ("what-if: center members reaching Materials (workbook)", CENTER_MAT_W, 3, 0),
    ("what-if: joint reviews (workbook)", len(fundamental_cycles(CLINKS_C)), 1, 0),
]
ok, width = 0, max(len(name) for name, *_ in checks)
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol + 1e-9 else "FAIL"; ok += status == "PASS"
    print(f"{name:{width}s} {got:10.4f}   book {want:9.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 10 adds the stocks. The capital architecture equips the components through which capital enters,
resides and leaves with conservation laws along the arrows: the carriers of capital are components of this chapter's
architecture, conversions run along its arrows, and its scoped constraints, floors, budgets and ratios, are governed by
the same tree-versus-cycle discipline. AXIOM-09 runs the same model interactively (Lab 1.9); this notebook and the
workbook are its reproducible record.